In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/shakesphere/First Citizen.txt


In [2]:
## Reading the data 
txt_data = "/kaggle/input/shakesphere/First Citizen.txt"

with open(txt_data, 'r', encoding = 'utf-8') as f:
    text = f.read()

In [3]:
## Getting vocab size and the character vocabulary
characters = sorted(list(set(text)))
vocab_size = len(characters)
print(f"Vocab size: {vocab_size}")
print(''.join(characters))

Vocab size: 65

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [4]:
## String to integer and integer to string conversion

stoi = {ch:i for i,ch in enumerate(characters)}
itos = {i:ch for i,ch in enumerate(characters)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda n: ''.join([itos[i] for i in n])

print(encode("What is up my guy"))
print(decode(encode("What is up my guy")))

[35, 46, 39, 58, 1, 47, 57, 1, 59, 54, 1, 51, 63, 1, 45, 59, 63]
What is up my guy


In [5]:
## Convert tokenized data into a tensor
import torch 
data = torch.tensor(encode(text), dtype = torch.long)
print(data.shape, data.dtype)

torch.Size([1115393]) torch.int64


In [6]:
## Split data into training and validation sets

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
len(train_data)

1003853

In [7]:
## Defining the maximum input size
block_size = 8
train_data[:block_size + 1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [8]:
## Splitting input and target
x = train_data[:block_size]
y = train_data[1:block_size+1]
for i in range(block_size):
    input_chars = x[:i+1]
    target = y[i]
    print(f"Input: {input_chars} and target is {target}")

Input: tensor([18]) and target is 47
Input: tensor([18, 47]) and target is 56
Input: tensor([18, 47, 56]) and target is 57
Input: tensor([18, 47, 56, 57]) and target is 58
Input: tensor([18, 47, 56, 57, 58]) and target is 1
Input: tensor([18, 47, 56, 57, 58,  1]) and target is 15
Input: tensor([18, 47, 56, 57, 58,  1, 15]) and target is 47
Input: tensor([18, 47, 56, 57, 58,  1, 15, 47]) and target is 58


In [9]:
torch.manual_seed(885)
batch_size = 4 # independent sequences processed in parallel
block_size = 8 # maximum characters at once

def get_batch(split):
    # create a batch of data of inputs and targets
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')

for batch in range(batch_size):
    for t in range(block_size):
        context = xb[batch, :t+1]
        target = yb[batch, t]
        print(f'content = {context.tolist()} target = {target}')

content = [53] target = 57
content = [53, 57] target = 58
content = [53, 57, 58] target = 1
content = [53, 57, 58, 1] target = 46
content = [53, 57, 58, 1, 46] target = 39
content = [53, 57, 58, 1, 46, 39] target = 60
content = [53, 57, 58, 1, 46, 39, 60] target = 43
content = [53, 57, 58, 1, 46, 39, 60, 43] target = 1
content = [1] target = 54
content = [1, 54] target = 53
content = [1, 54, 53] target = 53
content = [1, 54, 53, 53] target = 56
content = [1, 54, 53, 53, 56] target = 1
content = [1, 54, 53, 53, 56, 1] target = 50
content = [1, 54, 53, 53, 56, 1, 50] target = 47
content = [1, 54, 53, 53, 56, 1, 50, 47] target = 44
content = [15] target = 17
content = [15, 17] target = 10
content = [15, 17, 10] target = 0
content = [15, 17, 10, 0] target = 31
content = [15, 17, 10, 0, 31] target = 43
content = [15, 17, 10, 0, 31, 43] target = 39
content = [15, 17, 10, 0, 31, 43, 39] target = 56
content = [15, 17, 10, 0, 31, 43, 39, 56] target = 41
content = [39] target = 40
content = [39,

In [10]:
xb

tensor([[53, 57, 58,  1, 46, 39, 60, 43],
        [ 1, 54, 53, 53, 56,  1, 50, 47],
        [15, 17, 10,  0, 31, 43, 39, 56],
        [39, 40, 56, 43, 39, 57, 58,  6]])

In [11]:
yb

tensor([[57, 58,  1, 46, 39, 60, 43,  1],
        [54, 53, 53, 56,  1, 50, 47, 44],
        [17, 10,  0, 31, 43, 39, 56, 41],
        [40, 56, 43, 39, 57, 58,  6,  0]])

In [12]:
## BigramModel
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(885)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets  = None):
        logits = self.token_embedding_table(idx)
        if targets is None:
            loss = None
        else:
            #reshape for cross entropy compliance
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T) # flatten
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            # only last time step considered
            logits = logits[:, -1, :] # (B,C)
            probs = F.softmax(logits, dim = 1)
            # sample from the distribution, some randomnes
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1) 
            idx = torch.cat((idx, idx_next), dim = 1) # prediction appended to original tensor
        return idx
        
model = BigramLanguageModel(vocab_size)
output, loss = model(xb, yb)
loss

tensor(4.6682, grad_fn=<NllLossBackward0>)

In [14]:
context = torch.zeros((1, 1), dtype=torch.long)
print(decode(model.generate(context , max_new_tokens = 100)[0].tolist()))


gEdFL?uMxWRIw-Utlm
gl?wpI:s
IRl3RjQXug?MGngCXry,$&npA3L!-?mPBE.'!BeOilex;YLKUwHYxT!&Q!&Mjj;CKHH?,h.P


In [15]:
## instantiating the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-3)

In [16]:
batch_size = 3

for steps in range(10000):
    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none = True)

    loss.backward()
    optimizer.step()
print(loss.item())

2.3290135860443115


In [17]:
print(decode(model.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens = 300)[0].tolist()))



NALKMPr a w ke, n ithe y ore, tilurtl thinth ngraimofrod nk, KO:
Ye actiBr?


THU
DSYORB;3zje ou d d
P:
BowouVout BEYUSovol chrwalead lo IOre pMMy su'CHLO, HGHB-ha d ld fof

S:
MJ s in

se sav?
Whou ouite k? holo? adRYow, t so sthetranim, wn an
ahougeen phimirthierreinr n:
SThe fret s I thisis, h
'


In [18]:
batch_size = 32
block_size = 8
max_iters = 3000
eval_interval = 300
learning_rate = 1e-2
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200

In [19]:
def get_batch(split):
    # create a batch of data of inputs and targets
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [20]:
model = BigramLanguageModel(vocab_size)
m = model.to(device)

In [21]:
## function to calculate loss during training

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch(split)
            logits, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


In [25]:
model = BigramLanguageModel(vocab_size)
m = model.to(device)

## Updated training loop

optimizer  = torch.optim.AdamW(model.parameters(), lr = learning_rate)

for iter in range(max_iters):

    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: training loss = {losses['train']:.4f}, ")

    xb, yb = get_batch('train')

    ## evaluate loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none = True)
    loss.backward()
    optimizer.step()


step 0: training loss = 4.7107, 
step 300: training loss = 2.8193, 
step 600: training loss = 2.5500, 
step 900: training loss = 2.5032, 
step 1200: training loss = 2.4841, 
step 1500: training loss = 2.4793, 
step 1800: training loss = 2.4704, 
step 2100: training loss = 2.4744, 
step 2400: training loss = 2.4569, 
step 2700: training loss = 2.4502, 


In [26]:
context = torch.zeros((1,1), dtype = torch.long, device = device)
print(decode(model.generate(context, max_new_tokens = 300)[0].tolist()))


Bundin sines l, l yonghe k co end

hys VIOro ndy thealeithie my, r chare th, u, wre oulbeses,
Byours Inllaveathounte s wond l se, eevot jeaiorerokech&Card n thee;
BRofoulifre fr CUCly nl,
TCKERR:
t all!
Ligegew,
O:
Byownghinod, my heror, chourofaf wout t:

Thelinge.
RDURUpaneal bealdfeas!
TI henme w


In [38]:
## A mathematical trick for self attention 
torch.manual_seed(1337)
B, T, C = 4, 8, 2 # batch, time, channels
# batch - inputs stacked together
# time - inputs at different times, 1, 2, 3 in the sequence
# channels - each character is encoded to two numbers
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

In [42]:
xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t, c)
        xbow[b,t] = torch.mean(xprev, 0)

In [39]:
## A mathematical trick for self attention 
# perform self attention by averaging all the tokens prior to the current token

torch.manual_seed(42)
a = torch.tril(torch.ones(3,3))
a = a / torch.sum(a, 1, keepdim = True)
b = torch.randint(0, 10, (3, 2)).float()
c = a @ b

print('a = ')
print(a)
print('\n\n')
print('b = ')
print(b)
print('\n\n')
print('c = ')
print(c)
print('\n\n')

a = 
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])



b = 
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])



c = 
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])





In [43]:
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim = True)
xbow2 = wei @ x # batch matrix multiplaction
# multiplication would apply for all the batches
xbow2[0], xbow[0]

(tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]),
 tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]))

In [45]:
## final self attention version
tril = torch.tril(torch.ones(T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim = 1)
xbow3 = wei @ x
xbow3[0], xbow2[0]

(tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]),
 tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]))

In [49]:
## single self attention head

head_size = 16
key = nn.Linear(C, head_size, bias = False) # what i contain
query = nn.Linear(C, head_size, bias = False) # what i am looking for
value = nn.Linear(C, head_size, bias = False) # what i will communicate to you
k = key(x) # B, T, 16
q = query(x) # B, T, 16
wei = q @ k.transpose(-2, -1) * head_size**-0.5 # B,T,16 @ B, 16, T == B, T, T

tril = torch.tril(torch.ones(T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim = 1)
v = value(x)
out = wei @ v

wei[0]

tensor([[0.1254, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1253, 0.1507, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1259, 0.1665, 0.1627, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1262, 0.1783, 0.1613, 0.1873, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1248, 0.1354, 0.1672, 0.1993, 0.2181, 0.0000, 0.0000, 0.0000],
        [0.1238, 0.1146, 0.1705, 0.2063, 0.2692, 0.3177, 0.0000, 0.0000],
        [0.1256, 0.1579, 0.1637, 0.1923, 0.1797, 0.4031, 0.5253, 0.0000],
        [0.1230, 0.0967, 0.1746, 0.2148, 0.3331, 0.2792, 0.4747, 1.0000]],
       grad_fn=<SelectBackward0>)

In [92]:
batch_size = 32
block_size = 8
max_iters = 5000
eval_interval = 300
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 32

In [93]:
class Head(nn.Module):
    ## one head of self attention
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias = False)
        self.query = nn.Linear(n_embd, head_size, bias = False)
        self.value = nn.Linear(n_embd, head_size, bias = False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))


    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)

        # compute the attention scores
        wei = q @ k.transpose(-2, -1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim = 1)
        # calculate the value and then average the value with the wei (q * k)
        v = self.value(x)
        out = wei @ v
        return out

In [94]:
## multiple self attention heads

class multiheadselfattention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

    def forward(self, x):
        return torch.cat([h(x) for h in self.heads], dim = -1)

In [95]:
## feed forward class

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, n_embd),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)

In [98]:
## incoporating multi headed self attention into the bigram model 
## also add the feed forward network after self attention

class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.sa_heads = multiheadselfattention(4, n_embd//4)
        self.pos_embedding_table = nn.Embedding(block_size, n_embd)
        self.ffwd = FeedForward(n_embd)

    def forward(self, idx, targets  = None):
        B, T = idx.shape

        # idx and targets are both B,T tensors of integers
        tkn_embd = self.token_embedding_table(idx) # B T n_embd embedding layer
        pos_embd = self.pos_embedding_table(torch.arange(T, device = device)) # T, C
        x = tkn_embd + pos_embd # B, T, C the addition would be applied to all the batches
        x = self.sa_heads(x)
        x = self.ffwd(x)
        logits = self.lm_head(x) # B T vocab_size 
        
        if targets is None:
            loss = None
        else:
            #reshape for cross entropy compliance
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T) # flatten
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # only last time step considered
            logits = logits[:, -1, :] # (B,C)
            probs = F.softmax(logits, dim = 1)
            # sample from the distribution, some randomnes
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1) 
            idx = torch.cat((idx, idx_next), dim = 1) # prediction appended to original tensor
        return idx

In [99]:
model = BigramLanguageModel()
m = model.to(device)

## Updated training loop

optimizer  = torch.optim.AdamW(model.parameters(), lr = learning_rate)

for iter in range(max_iters):

    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: training loss = {losses['train']:.4f}, ")

    xb, yb = get_batch('train')

    ## evaluate loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none = True)
    loss.backward()
    optimizer.step()


step 0: training loss = 4.2205, 
step 300: training loss = 2.6260, 
step 600: training loss = 2.0738, 
step 900: training loss = 1.6467, 
step 1200: training loss = 1.3584, 
step 1500: training loss = 1.1692, 
step 1800: training loss = 1.0520, 
step 2100: training loss = 0.9551, 
step 2400: training loss = 0.8704, 
step 2700: training loss = 0.8008, 
step 3000: training loss = 0.7410, 
step 3300: training loss = 0.7009, 
step 3600: training loss = 0.6596, 
step 3900: training loss = 0.6459, 
step 4200: training loss = 0.5972, 
step 4500: training loss = 0.5763, 
step 4800: training loss = 0.5551, 


In [100]:
context = torch.zeros((1,1), dtype = torch.long, device = device)
print(decode(model.generate(context, max_new_tokens = 300)[0].tolist()))




Go oo't lore we dod ph, Rp:
Aue toh gosond seellun I mabr! ty Hat wil ghin; Bae surfo tou wil
Thanw a tonfde th oo YIN:
Ks sh Meass Va on AFire RINGINOds sin, you IIGLICKWAn Isealsn s, gh and noore.
AS:
GSUK:
Ham yoube I mayo IEC:
Dod.


Bsyshapt dber d.
G:
Frist AUPYnd.

KVargern
QL:
The youres,



In [107]:
# hyperparameters
batch_size = 64 # how many independent sequences will we process in parallel?
block_size = 256 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2

In [108]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

In [109]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [110]:
class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [111]:
## Transformer block

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [112]:
## GPT Language Model

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

        # better init, not covered in the original GPT video, but important, will cover in followup video
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [113]:
model = GPTLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

10.788929 M parameters


In [114]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


step 0: train loss 4.2579, val loss 4.2535
step 500: train loss 1.7247, val loss 1.8789
step 1000: train loss 1.3951, val loss 1.6157
step 1500: train loss 1.2721, val loss 1.5407
step 2000: train loss 1.1890, val loss 1.4974
step 2500: train loss 1.1282, val loss 1.4901
step 3000: train loss 1.0759, val loss 1.4941
step 3500: train loss 1.0211, val loss 1.5030
step 4000: train loss 0.9714, val loss 1.5055
step 4500: train loss 0.9182, val loss 1.5245
step 4999: train loss 0.8657, val loss 1.5623


In [115]:
# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))


Than is't predominy
Are mad-look'd with oaths a seep. Now ar I wonded,
And the way for the fair desure, that it is
Enforced with some justice, and thus hanging,
By if you find in arrival, it cryal
You shal see the struth of yourselves, that it was,
Your like a blessed praise have verlainly,
Opposing enough this oning on enemies!
Here of York and Enford, his orments
Was moveable lorder mest, when he is yours.

KING RICHARD II:
O worthy friend, I hate your success
Stands thus on.

QUEEN:
Your rigo


In [116]:
torch.save(m.state_dict(), "nano_gpt_shakespheare.pt")
